### sql連線測試

In [15]:
!pip install bcrypt

  Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl.metadata (10 kB)
Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl (150 kB)


In [6]:
from db import SqlUtils
db = SqlUtils()
from services.quest_service import teacher_show_quest
from services.teacher_service import login_teacher
from pprint import pprint

In [ ]:
email = 'test'
password = 'test'
teacher = login_teacher(email, password)
print(teacher)

登入成功，test 老師您好
{'teacher_id': 1, 'teacher_name': 'test', 'email': 'test', 'password_hash': '$2b$12$Yax4ueIMQqHkeLMPe6yQOeFefL9Xtf0BFBFheiwlLek0EvZ3v9Gve'}


In [9]:
result = teacher_show_quest(teacher)
menu_dict = {}
i = 1
# is_active 任務是否啟用
for quest in result:
    menu_dict[f'{i}'] = {
        'quest_id': quest['任務編號'],
        'quest_name': quest['任務名稱'],
        'is_active': quest['任務是否啟用']
    }
    i += 1
pprint(menu_dict, sort_dicts=False)
print(type(menu_dict['1']['is_active']))

{'1': {'quest_id': 1, 'quest_name': '今日初次練習', 'is_active': 1},
 '2': {'quest_id': 2, 'quest_name': '今日練習三次', 'is_active': 1},
 '3': {'quest_id': 3, 'quest_name': '本週練習三天', 'is_active': 1},
 '4': {'quest_id': 4, 'quest_name': '本週挑戰三首曲目', 'is_active': 1},
 '5': {'quest_id': 5, 'quest_name': '持之以恆', 'is_active': 1},
 '6': {'quest_id': 6, 'quest_name': '練習達人', 'is_active': 1},
 '7': {'quest_id': 7, 'quest_name': '歡迎回來', 'is_active': 1}}
<class 'int'>


In [ ]:
def stop_quest(teacher):
    result = teacher_show_quest(teacher)
    if isinstance(result, str):
        return result
    menu_dict = {}
    i = 1
    for quest in result:
        menu_dict[f'{i}'] = {
            'quest_id': quest['任務編號'],
            'quest_name': quest['任務名稱'],
            'is_active': quest['任務是否啟用']
        }
        i += 1
    while True:
        print('===== 需要停用/啟用哪一項任務 =====')
        for i in range(1,len(menu_dict)+1):
            quest = menu_dict[str(i)]
            if quest['is_active'] == 1:
                status = '啟用中'
            else:
                status = '停用中'
            print(f"[{i}] {quest['quest_name']} | {status}")

        choice = input('[0]取消\n:')
        if choice == '0':
            return '已取消變更'
        elif choice in menu_dict:
            select_quest = menu_dict[choice]
            change_quest = select_quest['quest_name']
            quest_id = select_quest['quest_id']
            if select_quest['is_active'] == 1:
                status = '啟用中'
            else:
                status = '停用中'
            while True:
                print(f'任務 [{change_quest}] 現在是{status}狀態\n是否變更 ?')
                choice = input('[1]是 [2]否')
                if choice == '1':
                    if select_quest['is_active'] == 1:
                        change_active = 0
                    else:
                        change_active = 1
                    sql = """
                    UPDATE quest_list SET is_active = %s WHERE quest_id = %s
                    """
                    result = db.execute(sql, (change_active, quest_id,))
                    if result > 0:
                        return '已變更狀態'
                    return '變更失敗'
                elif choice == '2':
                    break
                else:
                    print('輸入錯誤')
        else:
            print('輸入錯誤')


        
result = stop_quest(teacher)
print(result)



===== 需要停用/啟用哪一項任務 =====
[1] 今日初次練習
[2] 今日練習三次
[3] 本週練習三天
[4] 本週挑戰三首曲目
[5] 持之以恆
[6] 練習達人
[7] 歡迎回來


任務 [今日初次練習] 現在是停用中狀態
是否變更 ?
已變更狀態


登入成功，test 老師您好


ok
